# Evaluation and Reliability

Evaluation turns an agent requirement into repeatable examples, scoring rules, and release decisions.


## What to learn

- Outcome checks whether the final answer or action is correct.
- Process checks whether the agent used tools and permissions appropriately.
- Deterministic checks are best for facts, schemas, and executable behavior.
- Model or human judges help with qualities that require judgment.

## Decision rule

Start with real failures and a small representative dataset. Use multiple runs for variable behavior, calibrate subjective judges against people, report uncertainty, and block releases only on metrics tied to product risk.


## Runnable example

Run the following cells, then change one input and observe what changes. Focus on the decision being demonstrated rather than framework-specific syntax.


In [ ]:
"""Minimal eval harness: property-based checks + multi-run consistency.

No API needed — we simulate a non-deterministic model to show the pattern.
Swap `fake_model` for a real client in practice.
"""
# Import the dependencies used by this example.
import json
import random


# Define the data shape and small operations before running them.
def fake_model(prompt: str, seed: int) -> str:
    """Simulates non-determinism: same prompt, varying output."""
    rng = random.Random(seed)
    themes = ["onboarding friction", "pricing confusion"]
    if rng.random() < 0.15:  # occasional failure: invalid JSON
        return "The main themes are onboarding and pricing."
    picked = themes[: rng.randint(1, 2)]
    return json.dumps({"themes": picked, "n_respondents": 12})


def check_properties(raw: str) -> list[str]:
    """Property-based assertions: invariants, not exact strings."""
    failures = []
    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        return ["output is not valid JSON"]
    if not isinstance(data.get("themes"), list) or not data["themes"]:
        failures.append("themes must be a non-empty list")
    if not isinstance(data.get("n_respondents"), int) or data["n_respondents"] < 0:
        failures.append("n_respondents must be a non-negative int")
    return failures


def run_eval(prompt: str, runs: int = 10, pass_threshold: float = 0.8) -> dict:
    """Multiple-run consistency: gate on pass rate, not a single run."""
    results = [check_properties(fake_model(prompt, seed=i)) for i in range(runs)]
    passes = sum(1 for f in results if not f)
    pass_rate = passes / runs
    return {
        "pass_rate": pass_rate,
        "gate": "PASS" if pass_rate >= pass_threshold else "FAIL",
        "sample_failures": [f for f in results if f][:3],
    }


report = run_eval("Extract themes from these interview transcripts as JSON.")
print(json.dumps(report, indent=2))


## Study questions

1. What problem does this pattern solve?
2. What is the simplest alternative?
3. Which failure would tell you that the pattern is implemented incorrectly?
4. What measurement would justify adding more complexity?

## Before using it

- [ ] The requirement and success measure are explicit.
- [ ] Inputs and outputs are validated.
- [ ] Failures, retries, and stopping conditions are bounded.
- [ ] Security and privacy match the consequences of the action.
- [ ] The behavior is tested on realistic examples.
